# Tutorial 14 — Paper clinic and peer review

[Lecture notes](../lectures/14_paper_clinic.md) · [Slides](../slides/14_paper_clinic.pptx) · [Exercise sheet](../exercises/14_paper_clinic.md)

**25 minutes in class, CPU, NumPy only.** You will take two systems' predictions on the same items and produce what a results paragraph needs: each system's macro-F1, an interval for the difference, a test of whether the image is used, an error table and a formatted results row. The predictions are synthetic: 2,000 items and eight emotion labels, built so that the two scores land near the Task 12 organisers' baselines. Nothing here measures a real model.

| Minutes | Step |
|---|---|
| 0–5 | Two systems, one set of items |
| 5–10 | Macro-F1 from the confusion matrix |
| 10–18 | Paired bootstrap for the difference |
| 18–25 | The image-use counts, the error table, the results row |
| After class | Your turn: two functions with tests |

In [1]:
import sys
import math
import numpy as np
rng = np.random.default_rng(42)
print("Python", sys.version.split()[0], "NumPy", np.__version__, "seed", 42)

Python 3.13.1 NumPy 2.1.3 seed 42


## 1. Two systems, one set of items

The labels are the eight Task 12 emotions with unequal frequencies. System A stands for a text-only run and system B for a multimodal run. Each item has an `ease` that both systems share, so they tend to fail on the same items, as real systems do. A wrong answer goes to the label's usual confusion 60% of the time (awe with contentment, fear with sadness) and to a random label otherwise.

The cut-offs 0.508 and 0.513 were chosen so that the scores come out near 0.550 and 0.535. Look at them: **B is built to be right slightly more often than A** (about 52.6% of items against 51.6%, before lucky slips).

In [2]:
EMOTIONS = ["amusement", "awe", "contentment", "excitement", "anger", "disgust", "fear", "sadness"]
K, n_items = len(EMOTIONS), 2000
prior = np.array([0.22, 0.18, 0.20, 0.08, 0.04, 0.06, 0.10, 0.12])
confusable = np.array([3, 2, 1, 0, 5, 4, 7, 6])     # amusement<->excitement, awe<->contentment, anger<->disgust, fear<->sadness
y = rng.choice(K, size=n_items, p=prior)
ease = rng.uniform(size=n_items)                    # shared by both systems: this is what makes the comparison paired

def predict(cut):
    right = 0.5 * ease + 0.5 * rng.uniform(size=n_items) < cut
    slip = np.where(rng.uniform(size=n_items) < 0.6, confusable[y], rng.choice(K, size=n_items, p=prior))
    return np.where(right, y, slip)

pred_a = predict(0.508)      # "text-only"
pred_b = predict(0.513)      # "multimodal"
print("items:", n_items, " label counts:", np.bincount(y, minlength=K))
print("both right:", int(((pred_a == y) & (pred_b == y)).sum()), " both wrong:", int(((pred_a != y) & (pred_b != y)).sum()),
      " only A right:", int(((pred_a == y) & (pred_b != y)).sum()), " only B right:", int(((pred_a != y) & (pred_b == y)).sum()))

items: 2000  label counts: [452 337 406 158  60 129 212 246]
both right: 783  both wrong: 537  only A right: 351  only B right: 329


Both systems are right, or both are wrong, on 1,320 of the 2,000 items. Only the other 680, where exactly one system is right, can separate them. Section 4 uses the same kind of count.

## 2. Macro-F1 from the confusion matrix

For each class, F1 = 2·TP / (2·TP + FP + FN). Macro-F1 is the plain mean over classes, so anger (4% of items) counts as much as amusement (22%). A class with no gold and no predicted items gets F1 = 0 here, which matters for small resamples.

In [3]:
def confusion(y_true, y_pred):
    return np.bincount(y_true * K + y_pred, minlength=K * K).reshape(K, K)     # rows: gold, columns: predicted

def per_class_f1(y_true, y_pred):
    C = confusion(y_true, y_pred)
    tp, denom = np.diag(C), C.sum(0) + C.sum(1)                                # denom = 2TP + FP + FN
    return np.where(denom > 0, 2 * tp / np.maximum(denom, 1), 0.0)

def macro_f1(y_true, y_pred):
    return float(per_class_f1(y_true, y_pred).mean())

def accuracy(y_true, y_pred):
    return float((y_true == y_pred).mean())

print(f"{'class':12s} {'gold':>5s} {'F1 A':>6s} {'F1 B':>6s}")
for k, name in enumerate(EMOTIONS):
    print(f"{name:12s} {int((y == k).sum()):5d} {per_class_f1(y, pred_a)[k]:6.3f} {per_class_f1(y, pred_b)[k]:6.3f}")
f1_a, f1_b = macro_f1(y, pred_a), macro_f1(y, pred_b)
print(f"macro-F1  A {f1_a:.3f}   B {f1_b:.3f}   B - A {f1_b - f1_a:+.3f}")
print(f"accuracy  A {accuracy(y, pred_a):.3f}   B {accuracy(y, pred_b):.3f}   B - A {accuracy(y, pred_b) - accuracy(y, pred_a):+.3f}")
half = n_items // 2
print(f"macro-F1 of A on each half: {macro_f1(y[:half], pred_a[:half]):.3f}, {macro_f1(y[half:], pred_a[half:]):.3f}"
      f"   mean of halves {(macro_f1(y[:half], pred_a[:half]) + macro_f1(y[half:], pred_a[half:])) / 2:.4f}   whole set {f1_a:.4f}")
assert 0.53 < f1_b < f1_a < 0.56

class         gold   F1 A   F1 B
amusement      452  0.623  0.624
awe            337  0.541  0.549
contentment    406  0.594  0.583
excitement     158  0.443  0.446
anger           60  0.458  0.447
disgust        129  0.563  0.527
fear           212  0.565  0.539
sadness        246  0.606  0.562
macro-F1  A 0.549   B 0.535   B - A -0.014
accuracy  A 0.567   B 0.556   B - A -0.011
macro-F1 of A on each half: 0.556, 0.542   mean of halves 0.5489   whole set 0.5490


B was built to be right more often, and its measured macro-F1 is 0.014 lower. Which classes carry that gap, and how many gold items do they have?

The last line shows why macro-F1 has no per-item score to average: the mean of the two halves is not the score of the whole set. Accuracy is a mean over items. Macro-F1 is a ratio of counts per class, then a mean over classes. An interval for it has to recompute the whole metric on each resample.

## 3. Paired bootstrap for the difference

Draw 2,000 item indices with replacement. Score **both** systems on that same draw and keep the difference. Repeat 1,000 times. The middle 95% of the differences is the interval.

Before you run the cell: will the interval contain zero?

In [4]:
n_boot = 1000
diffs = np.empty(n_boot)
for i in range(n_boot):
    idx = rng.integers(0, n_items, n_items)                   # one draw, used for both systems
    diffs[i] = macro_f1(y[idx], pred_b[idx]) - macro_f1(y[idx], pred_a[idx])
lo, hi = np.quantile(diffs, [0.025, 0.975])
print(f"observed B - A: {f1_b - f1_a:+.3f}")
print(f"paired 95% interval: [{lo:+.3f}, {hi:+.3f}]   width {hi - lo:.3f}")
print(f"share of resamples where B beats A: {(diffs > 0).mean():.3f}")

unpaired = np.empty(n_boot)
for i in range(n_boot):
    ia, ib = rng.integers(0, n_items, n_items), rng.integers(0, n_items, n_items)   # the mistake: separate draws
    unpaired[i] = macro_f1(y[ib], pred_b[ib]) - macro_f1(y[ia], pred_a[ia])
ulo, uhi = np.quantile(unpaired, [0.025, 0.975])
print(f"unpaired 95% interval: [{ulo:+.3f}, {uhi:+.3f}]   width {uhi - ulo:.3f}")
assert lo < f1_b - f1_a < hi and lo < 0 < hi
assert (uhi - ulo) > (hi - lo)

observed B - A: -0.014
paired 95% interval: [-0.045, +0.014]   width 0.059
share of resamples where B beats A: 0.145
unpaired 95% interval: [-0.050, +0.023]   width 0.074


The interval contains zero. On these 2,000 items the data cannot tell the two systems apart, and the honest sentence is "B scored 0.014 lower; the 95% interval for the difference runs from −0.045 to +0.014". In this toy we know the truth: B's chance of being right on an item is about one point higher than A's. A paper that explained why "the image hurts" would be explaining noise.

The unpaired interval is wider because separate draws throw away the fact that both systems saw the same hard items. Always resample items once and score both systems on them.

## 4. Does the multimodal run use the image?

Task 12 gives every test row a decoy: the same caption with another painting. A multimodal run must do better with the right image, by a one-sided McNemar test at p < 0.05. The test looks only at items where the two conditions disagree on correctness.

Here the decoy run copies B, except on a random 15% of items where it is drawn again at a lower cut-off.

In [5]:
image_mattered = rng.uniform(size=n_items) < 0.15
pred_decoy = np.where(image_mattered, predict(0.45), pred_b)
real_right, decoy_right = pred_b == y, pred_decoy == y
b_count = int((real_right & ~decoy_right).sum())     # right only with the real image
c_count = int((~real_right & decoy_right).sum())     # right only with the decoy
print(f"macro-F1 with the real image {f1_b:.3f}, with the decoy {macro_f1(y, pred_decoy):.3f}")
print(f"right only with the real image: b = {b_count}    right only with the decoy: c = {c_count}")
print(f"labels unchanged by the decoy: {(pred_decoy == pred_b).mean():.3f}")

macro-F1 with the real image 0.535, with the decoy 0.519
right only with the real image: b = 70    right only with the decoy: c = 39
labels unchanged by the decoy: 0.924


If the image did nothing, each disagreement would be a coin flip between `b` and `c`. Is `b` far enough above `c`? You will compute the exact p-value in "Your turn" B.

The last line is the other half of the rule: a text-only run must give real and decoy rows the same label at least 99% of the time. A run that changes its label on 1 row in 10 is not text-only.

## 5. An error table, not a list of examples

Count every (gold, predicted) pair among B's errors and print the largest. Five rows with counts say more than five examples picked by eye.

In [6]:
C_b = confusion(y, pred_b)
errors_b = int(C_b.sum() - np.trace(C_b))
pairs = sorted(((int(C_b[g, p]), g, p) for g in range(K) for p in range(K) if g != p), reverse=True)
print(f"System B: {errors_b} errors on {n_items} items\n")
print("| Gold | Predicted | Count | Share of errors |")
print("|---|---|---:|---:|")
for count, g, p in pairs[:5]:
    print(f"| {EMOTIONS[g]} | {EMOTIONS[p]} | {count} | {count / errors_b:.1%} |")
top5 = sum(count for count, _, _ in pairs[:5])
print(f"\nTop five pairs: {top5} of {errors_b} errors ({top5 / errors_b:.1%})")
assert errors_b == int((pred_b != y).sum())

System B: 888 errors on 2000 items

| Gold | Predicted | Count | Share of errors |
|---|---|---:|---:|
| amusement | excitement | 138 | 15.5% |
| contentment | awe | 117 | 13.2% |
| awe | contentment | 114 | 12.8% |
| sadness | fear | 72 | 8.1% |
| fear | sadness | 63 | 7.1% |

Top five pairs: 504 of 888 errors (56.8%)


Five of the 56 possible confusion pairs hold 57% of the errors. In a paper, this table earns a sentence such as "awe and contentment account for 26% of all errors", followed by ten examples from that one pair, read by a speaker of the language. With real predictions, add a column per language.

## 6. The results row

Write the row from the variables, never by hand. A number typed into a table is a number with no run behind it.

In [7]:
row = {"system": "B: multimodal", "split": "dev (synthetic)", "metric": "macro-F1", "score": f1_b,
       "diff": f1_b - f1_a, "lo": lo, "hi": hi, "runs": "1 run, seed 42"}
print("| System | Split | Metric | Score | Diff. to A [95% CI] | Runs |")
print("|---|---|---|---:|---|---|")
print(f"| A: text-only | {row['split']} | {row['metric']} | {f1_a:.3f} | n/a | {row['runs']} |")
print(f"| {row['system']} | {row['split']} | {row['metric']} | {row['score']:.3f} | {row['diff']:+.3f} [{row['lo']:+.3f}, {row['hi']:+.3f}] | {row['runs']} |")
print("| B: multimodal | official test | macro-F1 | pending | n/a | frozen config |")
print()
print(f"B: multimodal & dev & {row['score']:.3f} & ${row['diff']:+.3f}$ [${row['lo']:+.3f}$, ${row['hi']:+.3f}$] \\\\")

| System | Split | Metric | Score | Diff. to A [95% CI] | Runs |
|---|---|---|---:|---|---|
| A: text-only | dev (synthetic) | macro-F1 | 0.549 | n/a | 1 run, seed 42 |
| B: multimodal | dev (synthetic) | macro-F1 | 0.535 | -0.014 [-0.045, +0.014] | 1 run, seed 42 |
| B: multimodal | official test | macro-F1 | pending | n/a | frozen config |

B: multimodal & dev & 0.535 & $-0.014$ [$-0.045$, $+0.014$] \\


The official row says "pending" until the organisers release scores. It never holds a development number.

## 7. Your turn

Replace each `raise` with your code, then run the test cell under it. Do not edit the tests.

**A. A paired bootstrap for any metric.** Wrap section 3 in a function, so that you can pass your own task's scorer (micro-F1 for Task 9, a per-language mean of macro-F1 for Task 12).

In [ ]:
# YOUR TURN
def paired_bootstrap(metric, y_true, pred_a, pred_b, n, rng):
    """95% interval (lo, hi) for metric(B) - metric(A) from n paired resamples of the items.
    `metric(y_true, y_pred)` returns one number. Draw one set of indices per resample and score both systems on it."""
    raise NotImplementedError("your turn")

In [9]:
lo_fn, hi_fn = paired_bootstrap(macro_f1, y, pred_a, pred_b, 1000, np.random.default_rng(0))
assert lo_fn < f1_b - f1_a < hi_fn
assert abs(lo_fn - lo) < 0.01 and abs(hi_fn - hi) < 0.01                     # agrees with section 3 up to resampling noise
assert paired_bootstrap(accuracy, y, pred_a, pred_a, 200, np.random.default_rng(0)) == (0.0, 0.0)   # a system against itself
acc_lo, acc_hi = paired_bootstrap(accuracy, y, pred_a, pred_b, 1000, np.random.default_rng(0))
print(f"macro-F1 difference: [{lo_fn:+.3f}, {hi_fn:+.3f}]   accuracy difference: [{acc_lo:+.3f}, {acc_hi:+.3f}]")
print("A passed")

macro-F1 difference: [-0.042, +0.015]   accuracy difference: [-0.035, +0.014]
A passed


**B. The exact one-sided McNemar test.** Under the null hypothesis each of the `b + c` disagreements is a fair coin flip. The p-value is the probability of `b` or more heads in `b + c` flips: the sum of C(b+c, k) / 2^(b+c) for k from b to b+c. Use `math.comb`.

In [ ]:
# YOUR TURN
def mcnemar_exact(b, c):
    """One-sided exact p-value that the first condition is better: P(X >= b) for X ~ Binomial(b + c, 0.5)."""
    raise NotImplementedError("your turn")

In [11]:
assert math.isclose(mcnemar_exact(7, 1), 9 / 256)          # (C(8,7) + C(8,8)) / 2^8
assert math.isclose(mcnemar_exact(5, 5), 638 / 1024)
assert mcnemar_exact(0, 0) == 1.0                          # no disagreements, no evidence
assert mcnemar_exact(1, 7) > 0.99                          # one-sided: the wrong direction is never significant
p_image = mcnemar_exact(b_count, c_count)
print(f"image-use check: b = {b_count}, c = {c_count}, one-sided p = {p_image:.4f}", "(passes)" if p_image < 0.05 else "(fails)")
print("B passed")

image-use check: b = 70, c = 39, one-sided p = 0.0019 (passes)
B passed


## Discussion

1. The interval in section 3 contains zero. Write the one sentence about B and A that your paper may say, and one sentence it may not.
2. The bootstrap resamples items. In Task 12 several annotations share one painting. What should you resample instead, and will the interval get wider or narrower?
3. Your error table shows that two labels are often confused. Name one explanation that blames the system and one that blames the labels.

## From the toy to your paper

Load your own development predictions for the baseline and the main system as three integer arrays (`y`, `pred_a`, `pred_b`) in place of section 1, and rerun sections 2, 3, 5 and 6. Task 12 ranks macro-F1 per language and then averaged, so pass a metric that does that. Task 6 teams keep one score per item (BERTScore F1) and bootstrap the mean. Paste the printed row into your paper and record the run in `experiment_log.md`.

Submit this notebook with both tests passing and a 150–250-word results paragraph for your own system that follows the four-part pattern from the lecture.